In [1]:
%reset -f

# This version uses pandas & matplotlib to provide interactive graphics
import pandas as pd
import numpy as np
from numpy.polynomial.polynomial import polyval
import array
import itertools
import matplotlib
%matplotlib inline
# matplotlib.use("nbagg")
import pickle

import matplotlib.pyplot as plt
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display
plt.ion

import gc
import psutil
import sys
import csv


In [2]:
# import the class AdmS for admissible constellations
from a20s_class import AdmS
# import the dictionaries for the constellations of interest
from a20s_class import J3dict
from a20s_class import J5dict

from a20s_class import J3keyarr
from a20s_class import J5keyarr


# Relative populations $w_{s,j}(37^\#)$
In this notebook we develop the models of relative populations of gaps $w_{s,j}(p_k^\#)$ in the cycles of gaps
$\mathcal{G}(p_k^\#)$.

We can create these models for all gaps $g < 2p_1$.  We have used $\mathcal{G}(37^\#)$
in our studies for the models of gaps.  Here we use $\mathcal{G}(19^\#)$, $\mathcal{G}(23^\#)$, or $\mathcal{G}(29^\#)$ when necessary, and then advance these models to $\mathcal{G}(37^\#)$ in order to align on $p_0=37$.
$$ w_{s,J}(p_k^\#) = w_{s,J}(\infty) - l_1 \prod_{p_1}^{p_k} \frac{p-J-2}{p-J-1} + l_2 \prod_{p_1}^{p_k} \frac{p-J-3}{p-J-1} - \ldots$$
with $l_j = L_j^T \cdot n_g(37^\#)/n_{2,1}(37^\#)$ when the span of $|s|<62$.  The left eigenvectors $L^T$ are an upper triangular Pascal matrix.

For the parameter $\lambda$ for the relative population models $w_g(\lambda) = w_g(p_k^\#)$ we use the subdominant eigenvalue
$$ \lambda = \lambda(p_k) = \prod_{p_1}^{p_k} \frac{p-J-2}{p-J-1} $$

For any constellations with span $|s|\ge 62$, the models will not be accurate.

The file 'a20s_class.py' contains the class AdmS(key,array) for admissible constellations, and this file contains
the dictionaries for our constellations of length J=3 or J=5.

We take initial counts from $\mathcal{G}(23^\#)$ and $\mathcal{G}(29^\#)$.
From the constraint $g < 2p_1$ we can only form exact models from this data for constellations with span 
$|s| \le 60$.

This is not quite general code.  We set bounds and sizes specific for $p_k=37$.  For example:  the maximum span for which we can develop the exact model $w_s(p^\#)$ is $|s|=60$.  For constellations beyond this span, the coefficients for the models are not accurate.  

In [3]:
# load the arrays for G(23#) and G(29#)
try:
    G23 = np.load("G23uint.npy")
except FileNotFoundError:
    print("File G23uint.npy not found.  Please run 01_Cycles.ipynb ")
    sys.exit()

try:
    rough29 = np.load("rough29bool.npy")
except FileNotFoundError:
    print("File rough29bool.npy not found.  Creating the array - ")
    rough29len = 1155*13*17*19*23*29
    rough29 = np.ones(rough29len, dtype=bool) # start all True
    primes29 = np.array([3,5,7,11,13,17,19,23,29], dtype='int')
    
    i=0,
    while (i < 9):
        p = primes29[i]
        j = (int)((p+1)/2 - 1)
        while (j < rough29len):
            rough29[j]=False
            j += p
        i += 1

    np.save("rough29bool.npy", rough29)


In [4]:
rough29[0:100], rough29[-100:]

(array([ True, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False,  True, False, False,
         True, False,  True,  True, False,  True, False, False,  True,
        False, False,  True,  True, False, False,  True, False,  True,
         True, False, False,  True, False,  True, False, False,  True,
        False, False, False,  True, False,  True,  True, False,  True,
         True, False,  True, False, False, False, False, False, False,
         True, False,  True, False, False,  True,  True, False, False,
        False, False,  True,  True, False, False,  True, False, False,
         True, False,  True, False, False,  True, False, False,  True,
         True, False, False, False, False,  True,  True, False,  True,
         True]),
 array([ True,  True, False,  True,  True, False, False, False, False,
         True,  True, False, False,  True, False, False,  True, False,
         True, False, False,  True, False, False,  True,  Tr

In [5]:
print(f"r29 len {len(rough29)} sum {sum(rough29)}")
# should be: len 3234846615 sum 1021870080

r29 len 3234846615 sum 1021870080


In [9]:
# block to check the available system memory
gc.collect()
memory = psutil.virtual_memory()
available_memory = memory.available
del memory
print(f"Available memory: {available_memory / (1024 ** 2):.2f} MB")

Available memory: 5426.08 MB


In [10]:
# 25x25 matrix of left eigenvectors -- an upper triangular Pascal matrix
# constellations up to |s|=60 have admissible driving terms of length at most j=J+24
eigLT=np.array([[1, 1, 1, 1, 1, 1, 1, 1, 1,  1,  1,  1,  1,   1,   1,   1,    1,    1,    1,     1,    1,    1,   1,    1,    1],
                [0, 1, 2, 3, 4, 5, 6, 7, 8,  9, 10, 11, 12,  13,  14,  15,   16,   17,   18,    19,   20,   21,  22,   23,   24],
                [0, 0, 1, 3, 6,10,15,21,28, 36, 45, 55, 66,  78,  91, 105,  120,  136,  153,   171,  190,  210, 231,  253,  276],
                [0, 0, 0, 1, 4,10,20,35,56, 84,120,165,220, 286, 364, 455,  560,  680,  816,   969, 1140, 1330, 1540, 1771, 2024],
                [0, 0, 0, 0, 1, 5,15,35,70,126,210,330,495, 715,1001,1365, 1820, 2380, 3060,  3876, 4845, 5985, 7315, 8855,10626],
                [0, 0, 0, 0, 0, 1, 6,21,56,126,252,462,792,1287,2002,3003, 4368, 6188, 8568, 11628,15504,20349,26334,33649,42504],
                [0, 0, 0, 0, 0, 0, 1, 7,28, 84,210,462,924,1716,3003,5005, 8008,12376,18564, 27132,38760,54264,74613,100947,134596],
                [0, 0, 0, 0, 0, 0, 0, 1, 8, 36,120,330,792,1716,3432,6435,11440,19448,31824, 50388,77520,116280,170544,245157,346104],
                [0, 0, 0, 0, 0, 0, 0, 0, 1,  9, 45,165,495,1287,3003,6435,12870,24310,43758, 75582,125970,203490,319770,490314,735471],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  1, 10, 55,220, 715,2002,5005,11440,24310,48620, 92378,167960,293930,497420,817190,1307504],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  1, 11, 66, 286,1001,3003, 8008,19448,43758, 92378,184756,352716,646646,1144066,1961256],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  1, 12,  78, 364,1365, 4368,12376,31824, 75582,167960,352716,705432,1352078,2496144],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  1,  13,  91, 455, 1820, 6188,18564, 50388,125970,293930,646646,1352078,2704156],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   1,  14, 105,  560, 2380, 8568, 27132,77520,203490,497420,1144066,2496144],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   1,  15,  120,  680, 3060, 11628,38760,116280,319770,817190,1961256],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   0,   1,   16,  136,  816,  3876,15504,54264,170544,490314,1307504],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   0,   0,    1,   17,  153,   969,4845,20349,74613,245157,735471],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   0,   0,    0,    1,   18,   171,1140,5985,26334,100947,346104],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   0,   0,    0,    0,    1,    19, 190, 1330, 7315,33649,134596],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   0,   0,    0,    0,    0,    1,   20,  210, 1540, 8855,42504],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   0,   0,    0,    0,    0,    0,    1,   21,  231, 1771,10626],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   0,   0,    0,    0,    0,    0,    0,    1,   22,  253, 2024],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   0,   0,    0,    0,    0,    0,    0,    0,    1,   23,  276],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   0,   0,    0,    0,    0,    0,    0,    0,    0,    1,   24],
                [0, 0, 0, 0, 0, 0, 0, 0, 0,  0,  0,  0,  0,   0,   0,   0,    0,    0,    0,    0,    0,    0,    0,    0,    1]])

In [11]:
# get the coefficients for the constellations with length J=3
try:
    # have the coefficients been previously computed and saved?
    with open("J3lj.pkl", 'rb') as fptr:
        J3ljs = pickle.load(fptr)
except FileNotFoundError:
    print("File J3lj.pkl not found!")
    # Create weights lj for each entry of J3dict
    J3ljs = {} # start with an empty dictionary
    p_init = 23
    wdenoms3 = (23-4)*(19-4)*(17-4)*(13-4)*(11-4)*(7-4)
    for key in J3dict:
        tests3 = AdmS(key, J3dict[key])
        spans3 = tests3.span()
        print(f"NEW_KEY {key} {J3dict[key]} span {spans3}")
        if (spans3 < 58):
            driverss3 = tests3.drivers(G23,True)
            print(f"{key} :\n n23 {driverss3[:]}")
            ws3 = driverss3[2:] / wdenoms3
            # print(f"{key} : winf {sum(ws3)}\n w23 {ws3}")
            jmax = len(ws3)-1
            while (ws3[jmax] <= 0.0 and jmax > 0):
                jmax -= 1
            if (jmax == 0):
                ljs3 = np.array([ws3[0]])
                J3ljs[key] = ljs3
            else:
                jmax += 1 # adjust jmax for end-of-array indexing
                ljs3 = np.dot(eigLT[0:jmax,0:jmax], ws3[0:jmax])
                jmax -= 1 # readjust jmax
            # print(f"{key} : jmax {jmax}\n ljs3(23#) {ljs3}")
            # advance the model from G(23#) to G(37#)
            j=1
            while (j <= jmax):
                # each factor is (p-4-j)/(p-4) for p=29,31,37
                ljs3[j] = ljs3[j]*((25-j)/25 * (27-j)/27 * (33-j)/33)
                j += 1

            # record the result in a dictionary
            print(f"{key} : jmax {jmax}\n ljs3(37#) {ljs3}")
            J3ljs[key] = ljs3
        else:
            # for constellations of span beyond 56, we use G(29#)
            print(f"NEW_KEY {key} has span {spans3} - Using G(29#)")
            driverss3 = tests3.driversbin(rough29,True)
            print(f"{key} :\n n29 {driverss3[:]}")
            ws3 = driverss3[2:] / (25*wdenoms3) # adding factor for p=29
            # print(f"{key} : winf {sum(ws3)}\n w23 {ws3}")
            jmax = len(ws3)-1
            while (ws3[jmax] <= 0.0 and jmax > 0):
                jmax -= 1
            if (jmax == 0):
                ljs3 = np.array([ws3[0]])
                J3ljs[key] = ljs3
            else:
                jmax += 1 # adjust jmax for end-of-array indexing
                ljs3 = np.dot(eigLT[0:jmax,0:jmax], ws3[0:jmax])
                jmax -= 1 # readjust jmax
            # print(f"{key} : jmax {jmax}\n ljs3(29#) {ljs3}")
            # advance the model from G(23#) to G(37#)
            j=1
            while (j <= jmax):
                # each factor is (p-4-j)/(p-4) for p=31,37
                ljs3[j] = ljs3[j]*((27-j)/27 * (33-j)/33)
                j += 1

            # record the result in a dictionary
            print(f"{key} : jmax {jmax}\n ljs3(37#) {ljs3}")
            J3ljs[key] = ljs3

    # save newly calculated J3ljs with pickle
    with open("J3lj.pkl", 'wb') as fptr:
        pickle.dump(J3ljs, fptr)

File J3lj.pkl not found!
NEW_KEY 242 [2, 4, 2] span 8
242 :
 n23 [     0.      0. 700245.      0.      0.      0.      0.      0.      0.
      0.      0.      0.      0.      0.      0.      0.      0.      0.
      0.      0.      0.      0.      0.      0.      0.      0.      0.
      0.      0.      0.]
242 : jmax 0
 ljs3(37#) [1.]
NEW_KEY 424 [4, 2, 4] span 10
424 :
 n23 [      0.       0. 1400490.       0.       0.       0.       0.       0.
       0.       0.       0.       0.       0.       0.       0.       0.
       0.       0.       0.       0.       0.       0.       0.       0.
       0.       0.       0.       0.       0.       0.]
424 : jmax 0
 ljs3(37#) [2.]
NEW_KEY 246 [2, 4, 6] span 12
246 :
 n23 [      0.       0. 1110186.  290304.       0.       0.       0.       0.
       0.       0.       0.       0.       0.       0.       0.       0.
       0.       0.       0.       0.       0.       0.       0.       0.
       0.       0.       0.       0.       0.       0.]


In [12]:
J3ljs

{'242': array([1.]),
 '424': array([2.]),
 '246': array([2.        , 0.37163778]),
 '426': array([1.        , 0.37163778]),
 '264': array([2.        , 0.37163778]),
 '626': array([1.33333333]),
 '662': array([1.33333333, 0.55745667]),
 '248': array([1.33333333, 0.55745667]),
 '2,10,2': array([2.66666667, 1.11491334]),
 '646': array([3.        , 1.48655113, 0.09723325]),
 '24,12': array([1.        , 0.74327556, 0.09723325]),
 '468': array([2.66666667, 1.67237002, 0.1944665 ]),
 '486': array([1.33333333, 0.55745667]),
 '2,10,6': array([2.        , 1.67237002, 0.29169976]),
 '666': array([2.        , 1.48655113, 0.1944665 ]),
 '686': array([3.33333333, 2.97310225, 0.58339951]),
 '12,12,12': array([2.        , 6.46752972, 8.28275777, 5.31084825, 1.76470902,
        0.2793215 , 0.01551515]),
 '12,18,12': array([8.00000000e+00, 3.24497298e+01, 5.46739955e+01, 4.94098212e+01,
        2.57601719e+01, 7.72769570e+00, 1.24424691e+00, 8.94645143e-02,
        1.67422883e-03]),
 '18,18,18': array([

In [13]:
# Repeat the above analysis for constellations of length J=5
# get the coefficients for the constellations with length J=3
try:
    # have the coefficients been previously computed and saved?
    with open("J5lj.pkl",'rb') as fptr:
        J5ljs = pickle.load(fptr)
except FileNotFoundError:
    print("File J5lj.pkl not found!")
    # Create weights lj for each entry of J3dict
    J5ljs = {} # start with an empty dictionary
    # We calculate populations in G(23#) if possible, balancing span (|s|<58) and speed
    wdenoms5 = (23-6)*(19-6)*(17-6)*(13-6)*(11-6)
    for key in J5dict:
        tests5 = AdmS(key, J5dict[key])
        spans5 = tests5.span()
        print(f"NEW_KEY {key} {J5dict[key]} span {spans5}")
        if (spans5 < 58):
            driverss5 = tests5.drivers(G23,True)
            print(f"{key} :\n n23 {driverss5[:]}")
            ws5 = driverss5[4:] / wdenoms5
            print(f"{key} : winf {sum(ws5)}\n w23 {ws5}")
            jmax = len(ws5)-1
            while (ws5[jmax] <= 0.0 and jmax > 0):
                jmax -= 1
            if (jmax == 0):
                ljs5 = np.array([ws5[0]])
                J5ljs[key] = ljs5
            else:
                jmax += 1 # adjust jmax for end-of-array indexing
                ljs5 = np.dot(eigLT[0:jmax,0:jmax], ws5[0:jmax])
                jmax -= 1 # readjust jmax
            print(f"{key} : jmax {jmax}\n ljs5(23#) {ljs5}")
            # advance the model from G(23#) to G(37#)
            j=1
            while (j <= jmax):
                # each factor is (p-6-j)/(p-6) for p=29,31,37
                ljs5[j] = ljs5[j]*((23-j)/23 * (25-j)/25 * (31-j)/31)
                j += 1

            # record the result in a dictionary
            print(f"{key} : jmax {jmax}\n ljs5(37#) {ljs5}")
            J5ljs[key] = ljs5
        else:
            # for constellations of span beyond 56, we use G(29#)
            print(f"NEW_KEY {key} has span {spans5} - Using G(29#)")
            driverss5 = tests5.driversbin(rough29,True)
            print(f"{key} :\n n29 {driverss5[:]}")
            ws5 = driverss5[4:] / (23*wdenoms5) # adding factor for p=29
            print(f"{key} : winf {sum(ws5)}\n w23 {ws5}")
            jmax = len(ws5)-1
            while (ws5[jmax] <= 0.0 and jmax > 0):
                jmax -= 1
            if (jmax == 0):
                ljs5 = np.array([ws5[0]])
                J5ljs[key] = ljs5
            else:
                jmax += 1 # adjust jmax for end-of-array indexing
                if (jmax <= 25):
                    ljs5 = np.dot(eigLT[0:jmax,0:jmax], ws5[0:jmax])
                else: # Too big for recorded eigenvectors
                    print(f"Warning - jmax {jmax} is too large for eigLT")
                    ljs5 = np.dot(eigLT, ws5[0:25])
                jmax -= 1 # readjust jmax
            # print(f"{key} : jmax {jmax}\n ljs5(29#) {ljs5}")
            # advance the model from G(23#) to G(37#)
            j=1
            while (j < len(ljs5)):
            # each factor is (p-6-j)/(p-6) for p=31,37
                ljs5[j] = ljs5[j]*((25-j)/25 * (31-j)/31)
                j += 1

            # record the result in a dictionary
            print(f"{key} : jmax {jmax}\n ljs5(37#) {ljs5}")
            J5ljs[key] = ljs5
            
    # save newly calculated J5ljs with pickle
    with open("J5lj.pkl", 'wb') as fptr:
        pickle.dump(J5ljs, fptr)            

File J5lj.pkl not found!
NEW_KEY 42424 [4, 2, 4, 2, 4] span 16
42424 :
 n23 [    0.     0.     0.     0. 85085.     0.     0.     0.     0.     0.
     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.]
42424 : winf 1.0
 w23 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0.]
42424 : jmax 0
 ljs5(23#) [1.]
42424 : jmax 0
 ljs5(37#) [1.]
NEW_KEY 2,10,2,10,2 [2, 10, 2, 10, 2] span 26
2,10,2,10,2 :
 n23 [     0.      0.      0.      0. 152544. 126240.  71280.      0.      0.
      0.      0.      0.      0.      0.      0.      0.      0.      0.
      0.      0.      0.      0.      0.      0.      0.      0.      0.
      0.      0.      0.]
2,10,2,10,2 : winf 4.114285714285714
 w23 [1.79284245 1.48369278 0.83775048 0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.  

In [14]:
# np.save('J5lj.npy',J5ljs)
# np.save('J3lj.npy',J3ljs)
J5ljs

{'42424': array([1.]),
 '2,10,2,10,2': array([4.11428571, 2.80738451, 0.65830974]),
 '2468,10': array([6.        , 8.18152058, 3.3738374 , 0.40941024]),
 '66266': array([4.57142857, 3.36886142, 0.65830974]),
 '6,12,2,6,12': array([4.92307692, 9.47158061, 6.28386566, 1.64128017, 0.12927826]),
 '64,14,42': array([3.6       , 4.41160424, 1.86521092, 0.27294016]),
 '6,12,10,66': array([ 4.96363636, 11.26162245,  9.47082178,  3.59371212,  0.58341433,
         0.02843844]),
 '10,2,10,2,10': array([6.10909091, 7.94088762, 3.04773026, 0.32752819]),
 '6,14,10,66': array([7.20000000e+00, 2.02492634e+01, 2.20961829e+01, 1.16796781e+01,
        3.01596952e+00, 3.27634534e-01, 8.60934297e-03]),
 '6,14,10,68': array([ 7.75384615, 22.93198672, 26.64658272, 15.39029961,  4.58642338,
         0.65386196,  0.03373457]),
 '10,20,10,20,10': array([8.00000000e+00, 5.28047151e+01, 1.54624618e+02, 2.64441653e+02,
        2.92617363e+02, 2.19289122e+02, 1.13238281e+02, 4.02317353e+01,
        9.66687248e+00, 

In [15]:
# we have the arrays of coefficients for the models derived from the eigenstructure
# associate the alternating signs with these coefficients --  
for key in J3ljs:
    if (len(J3ljs[key]) > 1):
        i = 1
        while (i < len(J3ljs[key])):
            J3ljs[key][i] *= -1
            i += 2


In [16]:
# enforcing the alternating signs for J=5
for key in J5ljs:
    if (len(J5ljs[key]) > 1):
        i = 1
        while (i < len(J5ljs[key])):
            J5ljs[key][i] *= -1
            i += 2


In [17]:
J3ljs

{'242': array([1.]),
 '424': array([2.]),
 '246': array([ 2.        , -0.37163778]),
 '426': array([ 1.        , -0.37163778]),
 '264': array([ 2.        , -0.37163778]),
 '626': array([1.33333333]),
 '662': array([ 1.33333333, -0.55745667]),
 '248': array([ 1.33333333, -0.55745667]),
 '2,10,2': array([ 2.66666667, -1.11491334]),
 '646': array([ 3.        , -1.48655113,  0.09723325]),
 '24,12': array([ 1.        , -0.74327556,  0.09723325]),
 '468': array([ 2.66666667, -1.67237002,  0.1944665 ]),
 '486': array([ 1.33333333, -0.55745667]),
 '2,10,6': array([ 2.        , -1.67237002,  0.29169976]),
 '666': array([ 2.        , -1.48655113,  0.1944665 ]),
 '686': array([ 3.33333333, -2.97310225,  0.58339951]),
 '12,12,12': array([ 2.        , -6.46752972,  8.28275777, -5.31084825,  1.76470902,
        -0.2793215 ,  0.01551515]),
 '12,18,12': array([ 8.00000000e+00, -3.24497298e+01,  5.46739955e+01, -4.94098212e+01,
         2.57601719e+01, -7.72769570e+00,  1.24424691e+00, -8.94645143e-02,

In [18]:
J5ljs

{'42424': array([1.]),
 '2,10,2,10,2': array([ 4.11428571, -2.80738451,  0.65830974]),
 '2468,10': array([ 6.        , -8.18152058,  3.3738374 , -0.40941024]),
 '66266': array([ 4.57142857, -3.36886142,  0.65830974]),
 '6,12,2,6,12': array([ 4.92307692, -9.47158061,  6.28386566, -1.64128017,  0.12927826]),
 '64,14,42': array([ 3.6       , -4.41160424,  1.86521092, -0.27294016]),
 '6,12,10,66': array([  4.96363636, -11.26162245,   9.47082178,  -3.59371212,
          0.58341433,  -0.02843844]),
 '10,2,10,2,10': array([ 6.10909091, -7.94088762,  3.04773026, -0.32752819]),
 '6,14,10,66': array([ 7.20000000e+00, -2.02492634e+01,  2.20961829e+01, -1.16796781e+01,
         3.01596952e+00, -3.27634534e-01,  8.60934297e-03]),
 '6,14,10,68': array([  7.75384615, -22.93198672,  26.64658272, -15.39029961,
          4.58642338,  -0.65386196,   0.03373457]),
 '10,20,10,20,10': array([ 8.00000000e+00, -5.28047151e+01,  1.54624618e+02, -2.64441653e+02,
         2.92617363e+02, -2.19289122e+02,  1.1323

## Preparing to plot $w_{s,J}(\lambda)$
The dictionary J3ljs contains the coefficients $l_j(s) = L_j^T \cdot w_s(37^\#)$.
Here in python, the dictionary of coefficients is indexed by the same keys as the defining dictionary.
The exact model $w_{s,J}(p^\#)$ for the relative population of $s$ in the cycle $\mathcal{G}(p^\#)$ is approximately
polynomial in $\lambda = \prod_{41}^{p_k} \frac{p-J-2}{p-J-1}$
$$ w_{s,J}(\lambda) \approx w_{s,J}(\infty) - l_2 \lambda + l_3 \lambda^2 - l_4 \lambda^3 + \cdots \; {\rm for} \; 0 < \lambda \le 1$$


In [19]:
# create an array of lambda values
lampar = np.arange(0,1,0.0005)
lampar = np.square(lampar) # These curves have more shape as lambda gets smaller, so we adjust the sampled values

In [20]:
len(J3ljs), len(J5ljs), lampar.size, len(lampar)

(21, 12, 2000, 2000)

In [21]:
# calculate values for the curves at the lambda values
ws3pk = {} # empty dictionary
for key in J3ljs:
    ws3pk[key] = np.zeros(lampar.size)
    i=0
    while (i < lampar.size):
        ws3pk[key][i] = polyval(lampar[i], J3ljs[key])
        i += 1


In [22]:
# calculate values for the curves for J=5
ws5pk = {} # empty dictionary
for key in J5ljs:
    ws5pk[key] = np.zeros(lampar.size)
    i=0
    while (i < lampar.size):
        ws5pk[key][i] = polyval(lampar[i], J5ljs[key])
        i += 1


In [23]:
ws3pk, ws5pk

({'242': array([1., 1., 1., ..., 1., 1., 1.], shape=(2000,)),
  '424': array([2., 2., 2., ..., 2., 2., 2.], shape=(2000,)),
  '246': array([2.        , 1.99999991, 1.99999963, ..., 1.6294763 , 1.62910512,
         1.62873376], shape=(2000,)),
  '426': array([1.        , 0.99999991, 0.99999963, ..., 0.6294763 , 0.62910512,
         0.62873376], shape=(2000,)),
  '264': array([2.        , 1.99999991, 1.99999963, ..., 1.6294763 , 1.62910512,
         1.62873376], shape=(2000,)),
  '626': array([1.33333333, 1.33333333, 1.33333333, ..., 1.33333333, 1.33333333,
         1.33333333], shape=(2000,)),
  '662': array([1.33333333, 1.33333319, 1.33333278, ..., 0.77754778, 0.77699102,
         0.77643398], shape=(2000,)),
  '248': array([1.33333333, 1.33333319, 1.33333278, ..., 0.77754778, 0.77699102,
         0.77643398], shape=(2000,)),
  '2,10,2': array([2.66666667, 2.66666639, 2.66666555, ..., 1.55509555, 1.55398203,
         1.55286796], shape=(2000,)),
  '646': array([3.        , 2.99999963, 

In [24]:
# Initialize master lists of colors
# we set colors by family, as determined by winf(s)

redlist=['#FF0000', '#AF3235', '#F282B4', '#EC008C', '#ED135A', '#E08080']
greenlist=['#00FF00', '#009B55', '#8DC73E', '#008B72', '#3FBC9D', '#A0D0A0']
bluelist=['#0000FF', '#00B0F0', '#00B3B8', '#006795', '#A0A0D0', '#46C5DD']
orangelist=['#F7965A', '#F26035', '#FAA21A', '#F89E7B', '#F0A080', '#D0B080']
misclist=['#02268F','#8C368C', '#AF72B0']
          


In [25]:

def draw_ws(xmax,yrng,J):
    # plotting the curves 
    plt.clf()
    fig, ax = plt.subplots()
    fig.set_size_inches(12,9)
    ax.set_title(f"$w_s(p_k^\#)$ for $p_0=37$ and select constellations $s$ of length J={J}")
    # set gridlines for x-axis (lambda)
    if (xmax < 0.05):
        marklampar = 0.001
    elif (xmax < 0.3):
        marklampar = 0.01
    elif (xmax < 0.75):
        marklampar = 0.05
    else:
        marklampar = 0.1
    ax.set_xticks(np.arange(0,xmax,marklampar))
    ax.grid(axis='x', color='#080408', lw=0.0625, markevery=marklampar)
    ax.grid(axis='y', color='#A9A9A9', lw=0.0625 )
    ax.set_xlim(0,xmax)
    ax.set_ylim(yrng[0],yrng[1])

    # We use key arrays to sort the graphs into a fixed order
    # And we use markers to cycle through the colors
    redmark = 0
    orangemark = 0
    bluemark = 0
    greenmark = 0
    miscmark = 0
    
    if (J == 3):
        i=0
        while (i < len(J3keyarr)):
            # we use the order of keys given in the key array
            curkey = J3keyarr[i]
            # color by winf
            winf = ws3pk[curkey][0]
            if (winf < 1.25):
                scolor = redlist[redmark]
                redmark += 1
                if (redmark >= len(redlist)): redmark=0
                linewid = 0.75 + 0.75*(redmark % 4)
            elif (winf < 1.8):
                scolor = orangelist[orangemark]
                orangemark += 1
                if (orangemark >= len(orangelist)): orangemark = 0
                linewid = 0.75 + 0.75*(orangemark % 4)
            elif (winf < 2.2):
                scolor = bluelist[bluemark]
                bluemark += 1
                if (bluemark >= len(bluelist)): bluemark = 0
                linewid = 0.75 + 0.75*(bluemark % 4)
            elif (winf < 3.1):
                scolor = greenlist[greenmark]
                greenmark += 1
                if (greenmark >= len(greenlist)): greenmark = 0
                linewid = 0.75 + 0.75*(greenmark % 4)
            else:
                scolor = misclist[miscmark]
                miscmark += 1
                if (miscmark >= len(misclist)): miscmark = 0
                linewid = 1.25
            # need flag for span < 62 -- accurate models
            span3s = sum(J3dict[curkey])
            if (span3s < 62):
                ax.plot(lampar, ws3pk[curkey], color=scolor, lw=linewid, label=str(curkey))
            else:
                ax.plot(lampar, ws3pk[curkey], color=scolor, lw=linewid, ls='-.', label=str(curkey))
            i += 1

        ax.legend(ncol=3)

    elif (J == 5):
        i=0
        while (i < len(J5keyarr)):
            # we use the order of keys given in the key array
            curkey = J5keyarr[i]
            # color by winf
            winf = ws5pk[curkey][0]
            if (winf < 1.25):
                scolor = redlist[redmark]
                redmark += 1
                if (redmark >= len(redlist)): redmark=0
                linewid = 0.75 + 0.75*(redmark % 4)
            elif (winf < 4.25):
                scolor = orangelist[orangemark]
                orangemark += 1
                if (orangemark >= len(orangelist)): orangemark = 0
                linewid = 0.75 + 0.75*(orangemark % 4)
            elif (winf < 5.5):
                scolor = bluelist[bluemark]
                bluemark += 1
                if (bluemark >= len(bluelist)): bluemark = 0
                linewid = 0.75 + 0.75*(bluemark % 4)
            elif (winf < 7.1):
                scolor = greenlist[greenmark]
                greenmark += 1
                if (greenmark >= len(greenlist)): greenmark = 0
                linewid = 0.75 + 0.5*(greenmark % 6)
            else:
                scolor = misclist[miscmark]
                miscmark += 1
                if (miscmark >= len(misclist)): miscmark = 0
                linewid = 1.25
            # need flag for span < 62 -- accurate models
            span5s = sum(J5dict[curkey])
            if (span5s < 62):
                ax.plot(lampar, ws5pk[curkey], color=scolor, lw=linewid, label=str(curkey))
            else:
                ax.plot(lampar, ws5pk[curkey], color=scolor, lw=linewid, ls='-.', label=str(curkey))

            i += 1
        ax.legend(ncol=2)

    plt.show()

# widgets for interacting with the graph of the w_g
xmaxSelect = widgets.SelectionSlider(options=[0.01,0.05,0.1,0.15,0.2,0.3,0.4,0.5,0.75,1.0],
                                      value=1.0, description="max lambda", layout=widgets.Layout(width='60%'), disabled=False)
yrngSelect = widgets.FloatRangeSlider(value=[0.0,8.0], min=0., max=8., step=0.25, orientation='horizontal',
                                      description="range w_s", layout=widgets.Layout(width='60%'), disabled=False)
JSelect = widgets.RadioButtons(options=[3,5], value=3, description='Length J', disabled=False) 

interact(draw_ws, xmax=xmaxSelect, yrng=yrngSelect, J=JSelect)


interactive(children=(SelectionSlider(description='max lambda', index=9, layout=Layout(width='60%'), options=(…

<function __main__.draw_ws(xmax, yrng, J)>